# Zulassungskarten

analysis of the results from pipeline.py

In [1]:
# imports
import json
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path

# Load & prepare Data

## Load

In [21]:
DATA_DIR = Path("../../data/03_processed") 

records = []

for file_path in DATA_DIR.glob("*.json"):
    with file_path.open(encoding="utf-8") as file:
        record = json.load(file)

    record["_file_name"] = file_path.name
    records.append(record)

df_processed = pd.json_normalize(records, sep="_")
df = df_processed.copy()

# df

## Helpers Functions

In [3]:
def parse_meters(value):
    if pd.isna(value):
        return float("nan")

    text = str(value).lower().replace("m", "").strip()
    if not text:
        return float("nan")

    # German decimal comma and thousands separators
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    else:
        text = text.replace(".", "")

    return pd.to_numeric(text, errors="coerce")

In [4]:
MONTH_NUMBERS = {
    "januar": 1,
    "februar": 2,
    "märz": 3,
    "april": 4,
    "mai": 5,
    "juni": 6,
    "juli": 7,
    "august": 8,
    "september": 9,
    "oktober": 10,
    "november": 11,
    "dezember": 12,
}


def parse_german_date(value):
    if pd.isna(value):
        return pd.NaT

    parts = str(value).strip().rstrip(".").split()
    if len(parts) != 3:
        return pd.NaT

    day = parts[0].rstrip(".")
    month = MONTH_NUMBERS.get(parts[1].lower())
    year = parts[2]

    if month is None:
        return pd.NaT

    return pd.to_datetime(
        f"{year}-{month:02d}-{day}",
        errors="coerce",
    )

## Prepare data

- join metadata with processed
- metadata columns are prefixed with `metadata_`; unprefixed data columns come from processed
- `metadata_available` marks rows with a matching metadata file
- sort by date or examinationnumber?

In [26]:
METADATA_DIR = Path("../../data/01_raw")

if "df_processed" not in globals():
    df_processed = pd.json_normalize(records, sep="_")

metadata_records = []

for file_path in METADATA_DIR.rglob("*.json"):
    with file_path.open(encoding="utf-8") as file:
        metadata_record = json.load(file)

    # Match the metadata file to its corresponding processed file.
    metadata_record["_file_name"] = file_path.name.replace(".json", "_processed.json")
    metadata_records.append(metadata_record)

df_metadata = pd.json_normalize(metadata_records, sep="_")
df_metadata = df_metadata.rename(
    columns={
        column: f"metadata_{column}"
        for column in df_metadata.columns
        if column != "_file_name"
    }
)

df = df_processed.merge(
    df_metadata,
    on="_file_name",
    how="left",
    validate="one_to_one",
    indicator="_metadata_join",
)
df["metadata_available"] = df.pop("_metadata_join").eq("both")

# df

In [27]:
df["examination_number_parsed"] = pd.to_numeric(df["examination_number"], errors="coerce").astype("Int64")
df["metadata_Prüfnummer_parsed"] = pd.to_numeric(df["metadata_Prüfnummer"], errors="coerce").astype("Int64")

df["metadata_Länge_parsed"] = df["metadata_Länge (in Meter)"].apply(parse_meters)
df["length_details_total_length_original_length_parsed"] = df["length_details_total_length_original_length"].apply(parse_meters)


df["examination_certificate_date_parsed"] = df["examination_certificate_date"].apply(parse_german_date) # nicht jedes wird geparsed aufgrund von schlechter ocr erkennung hmmm
df["metadata_Prüfdatum_parsed"] = pd.to_datetime(df["metadata_Prüfdatum"], dayfirst=True, errors="coerce") # hier auch extra pasred spalte?
df["metadata_year"] = df["metadata_Prüfdatum_parsed"].dt.year.astype("Int64")

In [28]:
with pd.option_context("display.max_columns", None):
    display(df)

,examination_number,stamp_text,origin_company,movie_title,movie_subtitle,produced_by,director,cinematography,set_design,cast_list,excerpt_note,intertitles,_file_name,length_details_individual_acts,length_details_total_length_original_length,length_details_total_length_length_after_cut,examination_certificate_examination_board,examination_certificate_location,examination_certificate_date,examination_certificate_decision_text,metadata_Titel_und_Signatur,metadata_Prüfnummer,metadata_Prüfdatum,metadata_Antragsteller,metadata_Produktion,metadata_Land,metadata_Filmart,metadata_Stumm-/Tonfilm,metadata_Länge (in Meter),metadata_Entscheidung,metadata_Unterlagenart,metadata_Benutzungsort,metadata_Bemerkung,metadata_Vorführungsbeschränkung,metadata_Ort des Prüfdatums,metadata_available,examination_number_parsed,metadata_Prüfnummer_parsed,metadata_Länge_parsed,length_details_total_length_original_length_parsed,examination_certificate_date_parsed,metadata_Prüfdatum_parsed,metadata_year
0,11751,Zulassungskarten für Bildstreifen sind öffentl...,"Universal Pictures Corp., New York",Liebestoll.,NaN,Deutsch-Nordische Film-Union G. m. b. H.,NaN,NaN,NaN,[],NaN,"[{'act_label': '1. Akt', 'text': '1. Akt. 1. E...",R 9346-I_7746_processed.json,"[{'act_number': 'I', 'original_length': '260 m...",498 m,NaN,Film-Prüfstelle Berlin,Berlin,13. November 1935,Der Bildstreifen wird zur öffentlichen Vorführ...,R 9346-I/7746\nLiebestoll.\n1920 - 1945,11751,13.11.1925,"Deutsch-Nordische Film-Union GmbH, Berlin","Universal-Pictures Corp., New York",USA,Spielfilm,Stummfilm,498,Jugendverbot,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,11751,11751,498.0,498.0,1935-11-13,1925-11-13,1925
1,38954,Zulassungskarten für Filme sind öffentliche Ur...,"Gebrüder Diehl-Film, Gräfeling bei München",Die Macht der Liebe.,Tobis zeigt einen Puppenfilm der Gebrüder Dieh...,Killerstraße 10/1.,NaN,Hans Asen,NaN,[],NaN,"[{'act_label': 'Fortsetzung', 'text': '2. Aha,...",R 9346-I_24154_processed.json,[],580 m,NaN,Film-Prüfstelle,Berlin,29. März 1935,Der Film wird zur öffentlichen Vorführung im D...,R 9346-I/24154\nDie Macht der Liebe\n1920 - 1945,38954,29.3.1935,"Gebr. Diehl, Gräfelfing b. München","Gebr. Diehl, Gräfelfing b. München",Deutschland,Spielfilm,Tonfilm,580,Jugendfrei,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,38954,38954,580.0,580.0,1935-03-29,1935-03-29,1935
2,16811,Zulassungskarten für Bildstreifen sind öffentl...,"Paramount-Film, U. S. A.",Ko Ko als Wedder.,Alfred Weiß zeigt Tintenmännchen im: Ko Ko als...,Universum-Film Aktiengesellschaft Berlin SW 68...,NaN,NaN,NaN,[],NaN,[],R 9346-I_11484_processed.json,[],181 m,NaN,Film-Prüfstelle Berlin,Berlin,3. Oktober 1927,Der Bildstreifen wird zur öffentlichen Vorführ...,R 9346-I/11484\nKo Ko als Wecker.\n1920 - 1945,16811,3.10.1927,"Universum-Film AG, Berlin","Paramount, New York",USA,Zeichentrickfilm,Stummfilm,181,Jugendfrei,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,16811,16811,181.0,181.0,1927-10-03,1927-10-03,1927
3,57539,NaN,"Tobis Filmkunst G. m. b. H., Berlin NW 7",Die Entlassung.,Tobis zeigt Emil Jannings in: Die Entlassung.,Friedrichstraße 100,Wolfgang Liebeneiner,Fritzz Wagner,Otto Hunte,"[{'role': 'Fürst Bismarck', 'actor': 'Emil Jan...",NaN,"[{'act_label': '1. Akt', 'text': '1. Vor dem A...",R 9346-I_35012_processed.json,"[{'act_number': '1. Rolle 1. Akt', 'original_l...",2991 m,NaN,Filmprüfstelle,Berlin,28. August 1942,Der Film wird zur öffentlichen Vorführung im D...,R 9346-I/35012\nDie Entlassung\n1920 - 1945,57539,28.8.1942,"Tobis Filmkunst GmbH, Berlin","Tobis Filmkunst GmbH, Berlin",Deutschland,Spielfilm,Tonfilm,2991,Jugendfrei vom 14. Lebensjahr ab,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,57539,57539,2991.0,2991.0,1942-08-28,1942-08-28,1942
4,15543,Zulassungskarten für Bildstreifen sind öffent....,"Berlin W 35, Genhinter Straße 32",Die Entstehung der Pelzmode.,Pinschewer-Film. 1. Frau Eva spricht zu ihrem ...,"Werbefilm G. m. b. H., Leitung J. Pinschewer",NaN,NaN,NaN,[],

In [15]:
df_look = df[df["examination_certificate_date_parsed"].isna()]

df_look = df_look[["examination_certificate_date_parsed", "examination_certificate_date"]]

df_look

,examination_certificate_date_parsed,examination_certificate_date
53,NaT,16 December 1933
164,NaT,NaN
198,NaT,NaN
300,NaT,19. Januar 1926; 03. September 1935
368,NaT,20. August 1932 / 27. Juni 1932
418,NaT,NaN
456,NaT,31. September 1928
492,NaT,8. April 1931 / 8. Januar 1936
495,NaT,1. April 1931; 3. Oktober 1935
513,NaT,NaN


## Evaluation

In [7]:
def compare_ocr_with_metadata(ocr_values, metadata_values):
    # Vergleicht die beiden Spalten zeilenweise.
    # Fehlende OCR-Werte sind bei vorhandenem Ground Truth-Wert falsch.
    # Fehlen die Metadaten, ist der Fall ohne weitere Prüfung unbekannt.
    ocr_values = pd.Series(ocr_values)
    metadata_values = pd.Series(metadata_values)

    if len(ocr_values) != len(metadata_values):
        raise ValueError("Die beiden Spalten müssen gleich lang sein.")

    original_index = ocr_values.index
    ocr_values = ocr_values.reset_index(drop=True)
    metadata_values = metadata_values.reset_index(drop=True)

    metadata_available = metadata_values.notna()
    both_values_present = metadata_available & ocr_values.notna()

    identical = pd.Series(pd.NA, index=ocr_values.index, dtype="boolean")
    identical.loc[metadata_available] = False
    identical.loc[both_values_present] = ocr_values.loc[both_values_present].eq(
        metadata_values.loc[both_values_present]
    )
    evaluated_count = int(metadata_available.sum())
    correct_count = int(identical.loc[metadata_available].sum())
    accuracy = correct_count / evaluated_count if evaluated_count else None
    accuracy_text = f"{accuracy:.2%}" if accuracy is not None else "n/a"
    unverified_count = int((~metadata_available).sum())
    ocr_without_metadata_count = int(
        (~metadata_available & ocr_values.notna()).sum()
    )

    print(
        f"{correct_count} von {evaluated_count} bewertbaren Zeilen korrekt "
        f"({accuracy_text})"
    )
    print(
        f"{unverified_count} Zeilen nicht bewertbar; davon "
        f"{ocr_without_metadata_count} mit OCR-Wert ohne Metadaten"
    )
    identical.index = original_index
    return identical


identical_examination_number = compare_ocr_with_metadata(
    df["examination_number"],
    df["metadata_Prüfnummer"],
)

# Manuell geprüfte Fälle ohne Metadaten hier eintragen.
manual_evaluation = {
    # "R 9346-I_123_processed.json": True,
    # "R 9346-I_456_processed.json": False,
}

manual_correct = df["_file_name"].map(manual_evaluation).astype("boolean")
evaluation = identical_examination_number.fillna(manual_correct)

evaluated_count = int(evaluation.notna().sum())
correct_count = int(evaluation.eq(True).sum())
accuracy = correct_count / evaluated_count if evaluated_count else None
accuracy_text = f"{accuracy:.2%}" if accuracy is not None else "n/a"
print(
    f"Nach manueller Bewertung: {correct_count} von {evaluated_count} "
    f"Zeilen korrekt ({accuracy_text})"
)

errors = evaluation.eq(False)

columns = [
    "examination_number",
    "metadata_Prüfnummer",
    "_file_name"
]
df_errors = df.loc[errors.fillna(False)][columns]
df_errors

2670 von 2799 bewertbaren Zeilen korrekt (95.39%)
25 Zeilen nicht bewertbar; davon 25 mit OCR-Wert ohne Metadaten
Nach manueller Bewertung: 2670 von 2799 Zeilen korrekt (95.39%)


,examination_number,metadata_Prüfnummer,_file_name
27,1999,4999,R 9346-I_2715_processed.json
31,1666,4666,R 9346-I_2459_processed.json
41,8378 / 122,8378,R 9346-I_5194_processed.json
87,4098,4093,R 9346-I_2031_processed.json
124,8607\n8608,8607,R 9346-I_5366_processed.json
...,...,...,...
2708,3901,3801,R 9346-I_1854_processed.json
2752,168 7462,7468,R 9346-I_4557_processed.json
2754,441 40,44140,R 9346-I_27057_processed.json
2785,601,661,R 9346-I_120_processed.json


In [22]:
# Länge (in Meter)

identical_length = compare_ocr_with_metadata(
    df["length_details_total_length_original_length"],
    df["metadata_Länge (in Meter)"]
)

identical_length

2343 von 2778 bewertbaren Zeilen korrekt (84.34%)
46 Zeilen nicht bewertbar; davon 42 mit OCR-Wert ohne Metadaten


0       True
1       True
2       True
3       True
4       True
        ... 
2819    True
2820    True
2821    True
2822    True
2823    True
Length: 2824, dtype: boolean

In [ ]:
# prüfdatum

# Analysis

## Intertitles

In [12]:
df_intertitles = pd.json_normalize(
    records,
    record_path="intertitles",
    meta=[
        "examination_number",
        "movie_title",
        "_file_name"
    ],
    errors="ignore",
    sep="_"
)

df_intertitles

,act_label,text,examination_number,movie_title,_file_name
0,Fortsetzung,gewerbes. d) Sängerurnde des Stuttgarter Falto...,14892,"Die Linotype-Setzmaschine.\nGeschichte, Fabrik...",R 9346-I_10000_processed.json
1,1. Akt,"1. Akt.\n1. Will Gallagher, ein junger verwege...",14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
2,1. Akt,gewandt werden. 8. Uns fehlen ein paar mutige ...,14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
3,2. Akt,1. Mein lieber Silberhörn ... jetzt fluttern w...,14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
4,Fortsetzung,lehrers. 10. Wozu brauchen wir einen Wunderheh...,14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
...,...,...,...,...,...
6128,3. Akt,"1. Du Bestie in Menschengestalt, Du Wölfin!!! ...",2,Die Wölfin.,R 9346-I_1_processed.json
6129,Fortsetzung,"nicht auch, Florence? 5. Ich fahre jetzt zur B...",2,Die Wölfin.,R 9346-I_1_processed.json
6130,3. Akt,"3. Akt. 1. Henry, Du versprachst mir, dich nic...",2,Die Wölfin.,R 9346-I_1_processed.json
6131,Fortsetzung,"Sprich die Wahrheit, wer hat mich ins Irrenhau...",2,Die Wölfin.,R 9346-I_1_processed.json


## Filmlength & cuts

In [19]:
df_filmlength = df.copy()

# todo: map date

# todo: map year

df_filmlength["original_length_m"] = (
    df_filmlength["length_details_total_length_original_length"].map(parse_meters)
)

df_filmlength["cut_length_m"] = (
    df_filmlength["length_details_total_length_length_after_cut"].map(parse_meters)
)
# auch hier individual acts angucken??

# ohne cuts haben keine cuts

df_filmlength["cut_length_available"] = (
    df_filmlength["original_length_m"].notna()
    & df_filmlength["cut_length_m"].notna()
)

valid_lengths = (
    df_filmlength["cut_length_available"] # beide längen enthalten
    & df_filmlength["original_length_m"].gt(0) # länge größer 0
    & df_filmlength["cut_length_m"].le(df_filmlength["original_length_m"]) # gekürtze känge ist kleiner oder gleich
)

df_filmlength["cut_meters"] = (
    df_filmlength["original_length_m"] - df_filmlength["cut_length_m"]
).where(valid_lengths)

df_filmlength["cut_rate"] = (
    df_filmlength["cut_meters"] / df_filmlength["original_length_m"]
).where(valid_lengths)

# filtert die zeilen direkt heraus
columns = [
    "_file_name",
    "movie_title",
    "original_length_m",
    "cut_length_m",
    "cut_meters",
    "cut_rate",
]
df_test = df_filmlength[df_filmlength["cut_length_m"].notna()][columns]

df_test

,_file_name,movie_title,original_length_m,cut_length_m,cut_meters,cut_rate
73,R 9346-I_1216_processed.json,Verlorene Einsichten.,1313.0,1299.00,14.00,0.010663
76,R 9346-I_2688_processed.json,Freie Bahn dem Künstler.,1098.0,1093.50,4.50,0.004098
111,R 9346-I_29488_processed.json,Der Kampf mit dem Moor. (Schmalfilm.),144.0,141.35,2.65,0.018403
150,R 9346-I_16150_processed.json,Mancher Irrsinn ist.,526.0,523.80,2.20,0.004183
189,R 9346-I_31128_processed.json,Zu den Güten der Sektion Schwaben des Deutsche...,441.0,449.00,NaN,NaN
...,...,...,...,...,...,...
2700,R 9346-I_22_processed.json,Die Banditen von Asnières.,2001.0,1956.00,45.00,0.022489
2720,R 9346-I_8325_processed.json,Bat und Patchadon am See.,2186.0,2168.70,17.30,0.007914
2725,R 9346-I_10585_processed.json,Die von der Straße leben. (Allegitim.),1949.0,1944.00,5.00,0.002565
2733,R 9346-I_24976_processed.json,Miva. -- Das Vermächtnis eines Missionars.,862.0,857.50,4.50,0.005220


### jährlich?

- 10494 ist die Karte mit dem Jahr 1972

In [ ]:
# yearly_length = (
#     df_filmlength.dropna(subset=["year"])
#     .groupby("year")
#     .agg(
#         records=("_file_name", "count"),
#         records_with_original_length=("original_length_m", "count"),
#         records_with_cut_length=("cut_length_available", "sum"),
#         films_with_cut=("has_cut", "sum"),
#         original_length_mean_m=("original_length_m", "mean"),
#         original_length_median_m=("original_length_m", "median"),
#         cut_meters_mean=("cut_meters", "mean"),
#         cut_meters_median=("cut_meters", "median"),
#         cut_rate_mean=("cut_rate", "mean"),
#         cut_rate_median=("cut_rate", "median"),
#     )
# )

# yearly_length["cut_share_percent"] = (
#     yearly_length["films_with_cut"]
#     .div(yearly_length["records_with_cut_length"])
#     .mul(100)
#     .where(yearly_length["records_with_cut_length"].gt(0))
# )

# yearly_length.round(2)


# hier nicht die raus filtern die keine cut length haben ig

plot

In [ ]:
plot_data = yearly_length[yearly_length["records_with_cut_length"] > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    yearly_length.index,
    yearly_length["original_length_median_m"],
    marker="o",
    label="Median der Originallänge",
)
axes[0].set_title("Filmlänge pro Prüfjahr")
axes[0].set_xlabel("Prüfjahr")
axes[0].set_ylabel("Meter")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(
    plot_data.index,
    plot_data["cut_share_percent"],
    marker="o",
    color="firebrick",
)
axes[1].set_title("Anteil gekürzter Filme (vergleichbare Datensätze)")
axes[1].set_xlabel("Prüfjahr")
axes[1].set_ylabel("Anteil in Prozent")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## next thing ig